In [1]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("/home/z5562390/work/dataset2023/japan_checkins.csv")

df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")


In [ ]:
origin_lat = df["latitude"].max()
origin_lon = df["longitude"].min()

METERS_PER_DEG_LAT = 111320

meters_per_deg_lon = 111320 * np.cos(np.radians(origin_lat))


In [ ]:

df["y_meters"] = (origin_lat - df["latitude"]) * METERS_PER_DEG_LAT


df["x_meters"] = (df["longitude"] - origin_lon) * meters_per_deg_lon


In [ ]:

GRID_SIZE = 500


df["grid_col"] = np.where(
    df["x_meters"].notna(),
    np.floor(df["x_meters"] / GRID_SIZE).astype("Int64") + 1,
    pd.NA
)


df["grid_row"] = np.where(
    df["y_meters"].notna(),
    np.floor(df["y_meters"] / GRID_SIZE).astype("Int64") + 1,
    pd.NA
)

df["grid_id"] = df["grid_row"].astype(str) + "_" + df["grid_col"].astype(str)


In [ ]:

# 00:00 -> 1
# 00:30 -> 2
# 01:00 -> 3
# ...
# 23:30 -> 48

df["time_block"] = np.where(
    df["timestamp"].notna(),
    (df["timestamp"].dt.hour * 2 + df["timestamp"].dt.minute // 30 + 1).astype("Int64"),
    pd.NA
)


In [ ]:

result = df[[
    "trail_id", "user_id", "venue_id",
    "latitude", "longitude", "timestamp",
    "grid_row", "grid_col", "grid_id", "time_block"
]]

print(result.head(10))

    trail_id  user_id  venue_id   latitude   longitude           timestamp  \
0   2013_802        9     19487  38.260281  140.852746 2013-03-06 02:20:00   
1   2013_802        9     16579  38.258552  140.853991 2013-03-06 09:32:00   
2  2013_1784       46      8592  34.666083  135.503343 2013-01-12 22:35:00   
3  2013_1784       46    167040        NaN         NaN 2013-01-12 23:15:00   
4  2013_1791       46       562  35.170296  136.882204 2013-03-24 02:20:00   
5  2013_1791       46     66151  35.315728  136.685550 2013-03-24 02:34:00   
6  2013_1791       46      4082  34.690005  135.518155 2013-03-24 04:07:00   
7  2013_1791       46       513  34.645717  135.514089 2013-03-24 06:29:00   
8  2013_1793       46       955  34.980764  135.747702 2013-03-28 08:09:00   
9  2013_1793       46      2393  34.994844  135.785001 2013-03-28 09:05:00   

  grid_row grid_col        grid_id time_block  
0   1617.0   2192.0  1617.0_2192.0          5  
1   1618.0   2192.0  1618.0_2192.0         20

In [ ]:

result.to_csv("processed_data.csv", index=False)

KeyboardInterrupt: 